In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [39]:
# 1. Đọc dữ liệu bằng pandas
df = pd.read_csv(Path("../data/diabetes_clean.csv"))

# 2. Xử lý giá trị bị thiếu (chọn 1 trong 2 phương án bên dưới):
# Option A: Điền giá trị 0 vào các ô bị trống (khuyên dùng nếu không muốn mất dòng)
df = df.fillna(0)

# Option B: Hoặc xóa luôn các dòng có dữ liệu bị thiếu (bỏ dấu # ở dòng dưới nếu chọn cách này)
# df = df.dropna()

# 3. Chuyển thành NumPy array để cắt lát X, y giữ nguyên code phía sau
data = df.to_numpy()

target_col = "Diabetes_binary"

X = df.drop(columns=[target_col]).to_numpy(dtype=float)
y = df[[target_col]].to_numpy(dtype=float)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (70841, 22)
y shape: (70841, 1)


In [40]:
np.random.seed(42)

indices = np.random.permutation(len(X))

split = int(0.8 * len(X))

train_idx = indices[:split]
test_idx = indices[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 56672
Testing samples: 14169


In [41]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0) + 1e-8

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

In [42]:
def relu(x):
    return np.maximum(0, x)


def relu_derivative(x):
    return (x > 0).astype(float)

In [43]:
def sigmoid(z):
    x = np.clip(z, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))


def sigmoid_derivative(a):
    return a * (1.0 - a)

In [44]:
# Step 7
np.random.seed(42)

input_dim = X.shape[1]

# Layer 1: 8 -> 32
W1 = np.random.randn(input_dim, 32) * np.sqrt(2.0 / input_dim)
b1 = np.zeros((1, 32))

# Layer 2: 32 -> 16
W2 = np.random.randn(32, 16) * np.sqrt(2.0 / 32)
b2 = np.zeros((1, 16))

# Layer 3: 16 -> 8
W3 = np.random.randn(16, 8) * np.sqrt(2.0 / 16)
b3 = np.zeros((1, 8))

# Layer 4: 8 -> 4
W4 = np.random.randn(8, 4) * np.sqrt(2.0 / 8)
b4 = np.zeros((1, 4))

# Layer 5: 4 -> 1 (Output)
W5 = np.random.randn(4, 1) * np.sqrt(2.0 / 4)
b5 = np.zeros((1, 1))

In [45]:
# Step 8
# giai thich phgan bay
def forward(X):
    # Layer 1
    z1 = X @ W1 + b1
    h1 = relu(z1)

    # Layer 2
    z2 = h1 @ W2 + b2
    h2 = relu(z2)

    # Layer 3
    z3 = h2 @ W3 + b3
    h3 = relu(z3)

    # Layer 4
    z4 = h3 @ W4 + b4
    h4 = relu(z4)

    # Layer 5 (Output)
    z5 = h4 @ W5 + b5
    y_hat = sigmoid(z5)

    cache = {
        "X": X,
        "z1": z1,
        "h1": h1,
        "z2": z2,
        "h2": h2,
        "z3": z3,
        "h3": h3,
        "z4": z4,
        "h4": h4,
        "z5": z5,
        "y_hat": y_hat,
    }
    return y_hat, cache

In [46]:
# Step 9
def binary_cross_entropy(y, y_hat):
    eps = 1e-8

    y_hat = np.clip(y_hat, eps, 1 - eps)
    loss = -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
    return loss

In [47]:
# Step 10
def backward(y, cache):
    X = cache["X"]
    z1, h1 = cache["z1"], cache["h1"]
    z2, h2 = cache["z2"], cache["h2"]
    z3, h3 = cache["z3"], cache["h3"]
    z4, h4 = cache["z4"], cache["h4"]
    yhat = cache["y_hat"]

    n = len(X)

    # ---------------------------
    # Layer 5 (Output Layer)
    # ---------------------------
    dz5 = (yhat - y) / n
    dW5 = h4.T @ dz5
    db5 = np.sum(dz5, axis=0, keepdims=True)

    # ---------------------------
    # Layer 4
    # ---------------------------
    dh4 = dz5 @ W5.T
    dz4 = dh4 * relu_derivative(z4)
    dW4 = h3.T @ dz4
    db4 = np.sum(dz4, axis=0, keepdims=True)

    # ---------------------------
    # Layer 3
    # ---------------------------
    dh3 = dz4 @ W4.T
    dz3 = dh3 * relu_derivative(z3)
    dW3 = h2.T @ dz3
    db3 = np.sum(dz3, axis=0, keepdims=True)

    # ---------------------------
    # Layer 2
    # ---------------------------
    dh2 = dz3 @ W3.T
    dz2 = dh2 * relu_derivative(z2)
    dW2 = h1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    # ---------------------------
    # Layer 1
    # ---------------------------
    dh1 = dz2 @ W2.T
    dz1 = dh1 * relu_derivative(z1)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    gradients = {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2,
        "dW3": dW3,
        "db3": db3,
        "dW4": dW4,
        "db4": db4,
        "dW5": dW5,
        "db5": db5,
    }

    return gradients


In [48]:
# # Step 11
# learning_rate = 0.01

# W1 -= learning_rate * gradients["dW1"]
# b1 -= learning_rate * gradients["db1"]
# W2 -= learning_rate * gradients["dW2"]
# b2 -= learning_rate * gradients["db2"]
# W3 -= learning_rate * gradients["dW3"]
# b3 -= learning_rate * gradients["db3"]

In [49]:
# Step 12
learning_rate = 0.01
epochs = 1000

for epoch in range(epochs):
    # Forward pass
    y_hat, cache = forward(X_train)

    # Compute loss
    loss = binary_cross_entropy(y_train, y_hat)

    # Backward pass
    gradients = backward(y_train, cache)

    # Update weights and biases
    W1 -= learning_rate * gradients["dW1"]
    b1 -= learning_rate * gradients["db1"]
    W2 -= learning_rate * gradients["dW2"]
    b2 -= learning_rate * gradients["db2"]
    W3 -= learning_rate * gradients["dW3"]
    b3 -= learning_rate * gradients["db3"]
    W4 -= learning_rate * gradients["dW4"]
    b4 -= learning_rate * gradients["db4"]
    W5 -= learning_rate * gradients["dW5"]
    b5 -= learning_rate * gradients["db5"]

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss:.4f}")

Epoch 100/1000, Loss: 0.6546
Epoch 200/1000, Loss: 0.6348
Epoch 300/1000, Loss: 0.6197
Epoch 400/1000, Loss: 0.6066
Epoch 500/1000, Loss: 0.5953
Epoch 600/1000, Loss: 0.5861
Epoch 700/1000, Loss: 0.5784
Epoch 800/1000, Loss: 0.5720
Epoch 900/1000, Loss: 0.5668
Epoch 1000/1000, Loss: 0.5625


In [50]:
# Step 13
y_prob, _ = forward(X_test)

y_pred = (y_prob >= 0.5).astype(int)

In [51]:
# Step 14
accuracy = np.mean(y_pred == y_test)

print("Accuracy:", accuracy)

Accuracy: 0.7143764556426


In [52]:
# Step 15
TP = np.sum((y_pred == 1) & (y_test == 1))
TN = np.sum((y_pred == 0) & (y_test == 0))
FP = np.sum((y_pred == 1) & (y_test == 0))
FN = np.sum((y_pred == 0) & (y_test == 1))

In [53]:
# giai thich ly thuyet
precision = TP / (TP + FP + 1e-8)
recall = TP / (TP + FN + 1e-8)

f1 = 2 * precision * recall / (precision + recall + 1e-8)

print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)

Precision: 0.707375478926356
Recall: 0.7865796831303601
F1-Score: 0.744878013042491
